In [32]:
import re
import numpy as np
import pandas as pd
df = pd.read_csv("hr_analytics_dataset.csv")

In [33]:
df.head(10)

,EmpID,Age,AgeGroup,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,RM297,18,18-25,Yes,Travel_Rarely,230,Research & Development,3,3,Life Sciences,...,3,80,0,0,2,3,0,0,0,0.0
1,RM302,18,18-25,No,Travel_Rarely,812,Sales,10,3,Medical,...,1,80,0,0,2,3,0,0,0,0.0
2,RM458,18,18-25,Yes,Travel_Frequently,1306,Sales,5,3,Marketing,...,4,80,0,0,3,3,0,0,0,0.0
3,RM728,18,18-25,No,Non-Travel,287,Research & Development,5,2,Life Sciences,...,4,80,0,0,2,3,0,0,0,0.0
4,RM829,18,18-25,Yes,Non-Travel,247,Research & Development,8,1,Medical,...,4,80,0,0,0,3,0,0,0,0.0
5,RM973,18,18-25,No,Non-Travel,1124,Research & Development,1,3,Life Sciences,...,3,80,0,0,5,4,0,0,0,0.0
6,RM1154,18,18-25,Yes,Travel_Frequently,544,Sales,3,2,Medical,...,3,80,0,0,2,4,0,0,0,0.0
7,RM1312,18,18-25,No,Non-Travel,1431,Research & Development,14,3,Medical,...,3,80,0,0,4,1,0,0,0,0.0
8,RM128,19,18-25,Yes,Travel_Rarely,528,Sales,22,1,Marketing,...,4,80,0,0,2,2,0,0,0,0.0
9,RM150,19,18-25,No,Travel_Rarely,1181,Research & Development,3,1,Medical,...,4,80,0,1,3,3,1,0,0,0.0


In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1480 entries, 0 to 1479
Data columns (total 38 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   EmpID                     1480 non-null   str    
 1   Age                       1480 non-null   int64  
 2   AgeGroup                  1480 non-null   str    
 3   Attrition                 1480 non-null   str    
 4   BusinessTravel            1480 non-null   str    
 5   DailyRate                 1480 non-null   int64  
 6   Department                1480 non-null   str    
 7   DistanceFromHome          1480 non-null   int64  
 8   Education                 1480 non-null   int64  
 9   EducationField            1480 non-null   str    
 10  EmployeeCount             1480 non-null   int64  
 11  EmployeeNumber            1480 non-null   int64  
 12  EnvironmentSatisfaction   1480 non-null   int64  
 13  Gender                    1480 non-null   str    
 14  HourlyRate         

In [37]:
df.isnull().sum()

empid                        0
age                          0
agegroup                     0
attrition                    0
businesstravel               0
dailyrate                    0
department                   0
distancefromhome             0
education                    0
educationfield               0
employeecount                0
employeenumber               0
environmentsatisfaction      0
gender                       0
hourlyrate                   0
jobinvolvement               0
joblevel                     0
jobrole                      0
jobsatisfaction              0
maritalstatus                0
monthlyincome                0
salaryslab                   0
monthlyrate                  0
numcompaniesworked           0
over18                       0
overtime                     0
percentsalaryhike            0
performancerating            0
relationshipsatisfaction     0
standardhours                0
stockoptionlevel             0
totalworkingyears            0
training

In [47]:
# 2. Chuyển toàn bộ tên cột sang chữ thường (lowercase)
df.columns = df.columns.str.lower()

In [45]:
# 3. Xóa các bản ghi trùng lặp mã nhân viên (empid), giữ lại bản ghi đầu
df = df.drop_duplicates(subset=['empid'], keep='first').reset_index(drop=True)

In [46]:
# 4. Chuẩn hóa giá trị không đồng nhất trong cột businesstravel
df['businesstravel'] = df['businesstravel'].replace(
    {'TravelRarely': 'Travel_Rarely'}
)

In [50]:
# 5. Xử lý giá trị thiếu (null) ở cột yearswithcurrmanager bằng trung vị (median)
# theo số năm làm việc (yearsatcompany)
df['yearswithcurrmanager'] = df.groupby('yearsatcompany')[
    'yearswithcurrmanager'
].transform(lambda x: x.fillna(x.median()))
df['yearswithcurrmanager'] = df['yearswithcurrmanager'].fillna(
    df['yearswithcurrmanager'].median()
)

In [51]:
# 6. Sửa lỗi logic: yearswithcurrmanager không thể lớn hơn yearsatcompany
df.loc[
    df['yearswithcurrmanager'] > df['yearsatcompany'], 'yearswithcurrmanager'
] = df['yearsatcompany']

In [52]:
# 7. Ép lại kiểu số nguyên (int) cho yearswithcurrmanager
df['yearswithcurrmanager'] = df['yearswithcurrmanager'].astype(int)

In [55]:
# 8. Loại bỏ các cột hằng số vô giá trị phân tích
redundant_cols = [
    # 1. Định danh cá nhân & Hằng số
    'empid',
    'employeenumber',
    'employeecount',
    'over18',
    'standardhours',
    # 2. Tỷ lệ lương giả lập gây nhiễu
    'dailyrate', 
    'hourlyrate', 
    'monthlyrate',
    # 3. Biến phân nhóm bị trùng lặp thông tin
    'agegroup',
    'salaryslab'
]
df = df.drop(columns=[col for col in redundant_cols if col in df.columns])

In [59]:
# 9. Biến danh mục không có thứ bậc (Nominal)
nominal_cols = [
    'businesstravel',
    'department',
    'educationfield',
    'gender',
    'jobrole',
    'maritalstatus',
]

In [60]:
# 10. Biến danh mục có thứ bậc (Ordinal Categorical)
ordinal_mapping = {
    'education': [1, 2, 3, 4, 5],
    'environmentsatisfaction': [1, 2, 3, 4],
    'jobinvolvement': [1, 2, 3, 4],
    'joblevel': [1, 2, 3, 4, 5],
    'jobsatisfaction': [1, 2, 3, 4],
    'performancerating': [3, 4],
    'relationshipsatisfaction': [1, 2, 3, 4],
    'stockoptionlevel': [0, 1, 2, 3],
    'worklifebalance': [1, 2, 3, 4],
}

In [62]:
# 11. Chuyển đổi các cột Nominal sang category
for col in nominal_cols:
    df[col] = df[col].astype('category')

In [63]:
# 12. Chuyển đổi các cột Original sang Categorical có thứ tự (Ordered Category)
for col, categories in ordinal_mapping.items():
    df[col] = pd.Categorical(df[col], categories = categories, ordered = True)

In [64]:
# 13. Mã hóa biến mục tiêu (Attrition) và Overtime sang dạng nhị phân (0/1)
df['attrition'] = df['attrition'].map({'No': 0, 'Yes': 1}).astype(int)
df['overtime'] = df['overtime'].map({'No': 0, 'Yes': 1}).astype(int)

In [68]:
# 14. Tối ưu bộ nhớ cho các cột số nguyên (Downcast int 64 -> int32 / int16)
int_cols = df.select_dtypes(include = ['int64']).columns
df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast = 'integer')

In [69]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   age                       1470 non-null   int8    
 1   attrition                 1470 non-null   int8    
 2   businesstravel            1470 non-null   category
 3   department                1470 non-null   category
 4   distancefromhome          1470 non-null   int8    
 5   education                 1470 non-null   category
 6   educationfield            1470 non-null   category
 7   environmentsatisfaction   1470 non-null   category
 8   gender                    1470 non-null   category
 9   jobinvolvement            1470 non-null   category
 10  joblevel                  1470 non-null   category
 11  jobrole                   1470 non-null   category
 12  jobsatisfaction           1470 non-null   category
 13  maritalstatus             1470 non-null   category
 14  mon

In [72]:
pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [74]:
from sqlalchemy import create_engine

# Step 1: Connect to PostgreSQL
# Replace placeholders with your actual details
username = "postgres"     # default user
password = "12345"
host = "localhost"
port = "5432"
database = "hr_db"        # Đã cập nhật tên database

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name = "hr_analytics_cleaned"  # Đã cập nhật tên bảng dữ liệu sạch
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'hr_analytics_cleaned' in database 'hr_db'.
